# 第6章 · 第2节：数据完整性 (Data Integrity)
## Cambridge International AS & A Level Computer Science (9618)

---

> **本节核心问题**：我们如何确保数据在输入和传输过程中是准确的、完整的、没有错误的？

**学习目标**：
1. 理解数据完整性的含义
2. 区分 **Validation（验证数据合理性）** 和 **Verification（验证数据准确性）**
3. 掌握各种 Validation 方法及其 Python 实现
4. 理解数据传输中的错误检测方法：奇偶校验、校验和

> 🏗️ **类比**：想象你在建造一座大楼。Security 是保证工地不被外人闯入（安全性），而 Integrity 是保证每块砖、每袋水泥都是合格的、每个尺寸都是准确的（完整性）。即使工地安全措施再好，如果建材有问题，大楼也会倒塌。

## 1. 什么是数据完整性？(What is Data Integrity?)

**数据完整性** 是指数据在其整个生命周期中保持 **准确性（accuracy）**、**一致性（consistency）** 和 **可靠性（reliability）** 的状态。

### 数据完整性面临的威胁

| 威胁 | 例子 |
|------|------|
| 人工输入错误 | 键盘输入错误：把 "2024" 打成 "2042" |
| 传输错误 | 网络传输中的比特翻转：1 变成了 0 |
| 软件bug | 程序错误导致数据被错误修改 |
| 硬件故障 | 硬盘坏道导致数据损坏 |
| 恶意篡改 | 黑客修改数据库中的记录 |

> 🧠 **思考**：数据完整性和数据安全性是互补的。安全性保护数据不被未授权的人访问或修改（外部威胁），而完整性确保数据本身是正确的（包括防止内部的无意错误）。

## 2. Validation vs Verification —— 最重要的区别！

这是本节 **最核心** 的概念，也是考试中最常出现的考点之一。

### 2.1 Validation（数据验证/校验）

**定义**：Validation 是由 **计算机自动执行** 的检查，用来确保输入的数据是 **合理的、符合规则的**。

> 📋 **关键词**：合理 (reasonable)、自动 (automatic)、规则 (rules)

**Validation 能做的**：检查数据是否在允许的范围内、格式是否正确、是否为空
**Validation 不能做的**：判断数据是否就是用户真正想要输入的

**举例**：如果要求输入年龄，Validation 可以检查输入是否在 0-150 之间。但如果用户本想输入 25 却误打成 52，Validation 无法发现这个错误——因为 52 也在合理范围内！

### 2.2 Verification（数据核验/核实）

**定义**：Verification 是用来确保数据是 **用户真正想要输入的**，通常需要 **人工参与**。

> 📋 **关键词**：准确 (accurate)、人工 (human)、意图 (intended)

**举例**：让用户输入两次邮箱地址并比较，或让用户目视检查屏幕上的信息是否正确。

### 对比表

| 特征 | Validation | Verification |
|------|-----------|-------------|
| 中文 | 数据验证/校验 | 数据核验/核实 |
| 执行者 | **计算机自动** | 通常需要 **人工参与** |
| 检查目标 | 数据是否 **合理** | 数据是否 **准确** |
| 问的问题 | "这个数据有意义吗？" | "这是你想输入的吗？" |
| 能否消除所有错误？ | ❌ 不能 | ❌ 不能（但能减少） |
| 例子 | 检查年龄是否在 0-150 之间 | 让用户输入两次密码并比较 |

> ⚠️ **考试警告**：很多学生会混淆这两个概念！记住这个口诀：
> - **Validation** = "数据合理吗？" （电脑自动检查）
> - **Verification** = "数据对吗？" （人来确认）

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 SimHei 字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DengXian', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('SimHei')
if 'SimHei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 SimHei 已加载')
else:
    print(f'⚠️ SimHei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['SimHei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 SimHei 字体文件')

import matplotlib.patches as patches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Validation side
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('Validation (数据验证)', fontsize=14, fontweight='bold', color='#2196F3')

# Input
ax1.text(5, 7, 'User Input: age = 52', ha='center', fontsize=11,
         bbox=dict(facecolor='lightyellow', edgecolor='orange', boxstyle='round,pad=0.5'))

# Computer check
comp_box = patches.FancyBboxPatch((1.5, 4), 7, 2, boxstyle="round,pad=0.3",
                                   facecolor='#E3F2FD', edgecolor='#2196F3', linewidth=2)
ax1.add_patch(comp_box)
ax1.text(5, 5.3, '🖥️ Computer Checks Automatically:', ha='center', fontsize=10, fontweight='bold')
ax1.text(5, 4.5, 'Is 52 between 0-150?  YES ✅', ha='center', fontsize=10)

ax1.annotate('', xy=(5, 6), xytext=(5, 6.7), arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Result
ax1.text(5, 3, 'Result: ACCEPTED', ha='center', fontsize=11, color='green', fontweight='bold')
ax1.text(5, 2, 'But user meant 25, not 52!', ha='center', fontsize=10, color='red', style='italic')
ax1.text(5, 1, 'Validation CANNOT catch this error', ha='center', fontsize=9, color='red',
         bbox=dict(facecolor='#FFEBEE', edgecolor='red', boxstyle='round'))

# Verification side
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('Verification (数据核验)', fontsize=14, fontweight='bold', color='#4CAF50')

# Double entry
ax2.text(5, 7, 'First entry:  alice@email.com', ha='center', fontsize=10,
         bbox=dict(facecolor='lightyellow', edgecolor='orange', boxstyle='round,pad=0.3'))
ax2.text(5, 6, 'Second entry: alice@emial.com', ha='center', fontsize=10,
         bbox=dict(facecolor='lightyellow', edgecolor='orange', boxstyle='round,pad=0.3'))

# Compare
comp_box2 = patches.FancyBboxPatch((1.5, 3.5), 7, 2, boxstyle="round,pad=0.3",
                                    facecolor='#E8F5E9', edgecolor='#4CAF50', linewidth=2)
ax2.add_patch(comp_box2)
ax2.text(5, 4.8, '👤 System Compares Two Entries:', ha='center', fontsize=10, fontweight='bold')
ax2.text(5, 4.0, '"email" ≠ "emial"  MISMATCH ❌', ha='center', fontsize=10)

ax2.annotate('', xy=(5, 5.5), xytext=(3, 5.8), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax2.annotate('', xy=(5, 5.5), xytext=(7, 5.8), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Result
ax2.text(5, 2.5, 'Result: REJECTED - Please re-enter', ha='center', fontsize=11, color='red', fontweight='bold')
ax2.text(5, 1.5, 'Verification CAN catch typos', ha='center', fontsize=10, color='green',
         bbox=dict(facecolor='#E8F5E9', edgecolor='green', boxstyle='round'))

plt.tight_layout()
plt.show()


## 3. Validation 方法详解 (Validation Methods)

以下是 A Level 考试要求掌握的各种 Validation 方法。每种方法我们都会用 Python 来实现。

### 3.1 Range Check（范围检查）

检查数据是否在 **允许的最小值和最大值之间**。

> 🎯 **用途**：适用于有明确上下限的数值数据
> **例子**：年龄必须在 0-150 之间，月份必须在 1-12 之间

### 3.2 Format Check（格式检查）

检查数据是否 **匹配预期的格式/模式**。通常使用正则表达式（Regular Expression, Regex）。

> 🎯 **用途**：适用于有特定格式要求的数据
> **例子**：邮箱地址必须包含 @ 符号，日期格式必须是 DD/MM/YYYY

### 3.3 Length Check（长度检查）

检查数据的 **字符数** 是否符合要求。

> 🎯 **用途**：适用于有固定长度的数据
> **例子**：中国手机号必须是 11 位，邮政编码必须是 6 位

### 3.4 Presence Check（存在性检查 / 非空检查）

检查必填字段 **是否已经填写**（不为空）。

> 🎯 **用途**：确保必填字段不被跳过
> **例子**：注册表单中的姓名、邮箱不能为空

### 3.5 Existence Check（查找检查）

检查输入的值是否 **存在于一个预定义的列表或数据库中**。

> 🎯 **用途**：确保输入的值是有效的选项
> **例子**：输入的国家代码必须是有效的 ISO 国家代码

### 3.6 Limit Check（限值检查）

类似范围检查，但只检查 **单边界限**（最大值或最小值）。

> 🎯 **用途**：只有上限或下限的情况
> **例子**：提款金额不能超过余额，考试分数不能为负数

### 3.7 Check Digit（校验位）

在数据末尾添加一个 **通过算法计算得出的额外数字**，用于检测数据输入或传输中的错误。

> 🎯 **用途**：检测转录错误（transposition errors）和输入错误
> **例子**：ISBN 编号、信用卡号码、身份证号最后一位

In [ ]:
import re

def range_check(value, min_val, max_val):
    # 范围检查 (Range Check)
    if min_val <= value <= max_val:
        return True, f"✅ {value} 在范围 [{min_val}, {max_val}] 内"
    return False, f"❌ {value} 不在范围 [{min_val}, {max_val}] 内"

def format_check(value, pattern, description):
    # 格式检查 (Format Check) - 使用正则表达式
    if re.match(pattern, value):
        return True, f"✅ '{value}' 匹配格式: {description}"
    return False, f"❌ '{value}' 不匹配格式: {description}"

def length_check(value, expected_length):
    # 长度检查 (Length Check)
    actual = len(str(value))
    if actual == expected_length:
        return True, f"✅ 长度为 {actual}，符合要求 ({expected_length})"
    return False, f"❌ 长度为 {actual}，不符合要求 ({expected_length})"

def presence_check(value):
    # 存在性检查 / 非空检查 (Presence Check)
    if value is not None and str(value).strip() != "":
        return True, f"✅ 字段已填写: '{value}'"
    return False, "❌ 字段为空！此字段必须填写"

def existence_check(value, valid_set):
    # 查找检查 (Existence Check)
    if value in valid_set:
        return True, f"✅ '{value}' 存在于有效列表中"
    return False, f"❌ '{value}' 不存在于有效列表中"

def limit_check(value, limit, check_type="max"):
    # 限值检查 (Limit Check)
    if check_type == "max":
        if value <= limit:
            return True, f"✅ {value} <= {limit} (未超过上限)"
        return False, f"❌ {value} > {limit} (超过上限!)"
    else:
        if value >= limit:
            return True, f"✅ {value} >= {limit} (未低于下限)"
        return False, f"❌ {value} < {limit} (低于下限!)"

# =================== 测试所有验证方法 ===================
print("=" * 60)
print("  Validation Methods Demo (数据验证方法演示)")
print("=" * 60)

# Range Check
print("\n📌 1. Range Check (范围检查)")
print("   场景: 学生年龄必须在 5-100 之间")
for age in [15, 3, 200, 50]:
    ok, msg = range_check(age, 5, 100)
    print(f"   输入 {age}: {msg}")

# Format Check
print("\n📌 2. Format Check (格式检查)")
print("   场景: 验证邮箱格式")
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
for email in ["alice@gmail.com", "bob@", "test.user@company.co.uk", "invalid"]:
    ok, msg = format_check(email, email_pattern, "email格式")
    print(f"   输入 '{email}': {msg}")

# Length Check
print("\n📌 3. Length Check (长度检查)")
print("   场景: 中国手机号必须是 11 位")
for phone in ["13812345678", "1381234", "138123456789"]:
    ok, msg = length_check(phone, 11)
    print(f"   输入 '{phone}': {msg}")

# Presence Check
print("\n📌 4. Presence Check (非空检查)")
print("   场景: 姓名字段不能为空")
for name in ["Alice", "", "   ", None, "Bob"]:
    ok, msg = presence_check(name)
    print(f"   输入 {repr(name)}: {msg}")

# Existence Check
print("\n📌 5. Existence Check (查找检查)")
print("   场景: 国家代码必须有效")
valid_countries = {"CN", "US", "UK", "JP", "DE", "FR", "AU"}
for country in ["CN", "US", "XX", "ZZ"]:
    ok, msg = existence_check(country, valid_countries)
    print(f"   输入 '{country}': {msg}")

# Limit Check
print("\n📌 6. Limit Check (限值检查)")
print("   场景: 提款金额不能超过余额 (余额: 1000)")
for amount in [500, 1000, 1500]:
    ok, msg = limit_check(amount, 1000, "max")
    print(f"   提款 {amount}: {msg}")

### 3.7 Check Digit 详解 —— 校验位算法

校验位是数据完整性中非常巧妙的发明。它通过数学计算在数据末尾添加一个额外的数字，使得如果数据中有任何一位被改变或两位被交换，都能被检测到。

#### ISBN-13 校验位算法

ISBN-13（国际标准书号，13位版本）的最后一位就是校验位。

**计算步骤**：
1. 将前 12 位数字从左到右分别乘以权重 1, 3, 1, 3, 1, 3, ... （交替）
2. 将所有乘积相加
3. 用总和对 10 取模（mod 10）
4. 校验位 = (10 - 余数) mod 10

**举例**：ISBN 978-0-306-40615-?

```
位置:     9   7   8   0   3   0   6   4   0   6   1   5
权重:     1   3   1   3   1   3   1   3   1   3   1   3
乘积:     9  21   8   0   3   0   6  12   0  18   1  15

总和 = 9 + 21 + 8 + 0 + 3 + 0 + 6 + 12 + 0 + 18 + 1 + 15 = 93
余数 = 93 mod 10 = 3
校验位 = (10 - 3) mod 10 = 7

完整 ISBN: 978-0-306-40615-7 ✅
```

In [ ]:
def isbn13_check_digit(isbn_12):
    # 计算 ISBN-13 校验位
    digits = [int(d) for d in str(isbn_12) if d.isdigit()]

    if len(digits) != 12:
        return None, "需要恰好12位数字"

    total = 0
    print("ISBN-13 校验位计算过程:")
    print("=" * 60)
    print(f"{'位置':>4s} {'数字':>4s} {'权重':>4s} {'乘积':>6s}")
    print("-" * 25)

    for i, d in enumerate(digits):
        weight = 1 if i % 2 == 0 else 3
        product = d * weight
        total += product
        print(f"{i+1:>4d} {d:>4d} {weight:>4d} {product:>6d}")

    print("-" * 25)
    print(f"{'总和':>14s} {total:>6d}")

    remainder = total % 10
    check_digit = (10 - remainder) % 10

    print(f"\n余数 = {total} mod 10 = {remainder}")
    print(f"校验位 = (10 - {remainder}) mod 10 = {check_digit}")

    return check_digit, "计算成功"

def verify_isbn13(isbn):
    # 验证完整的 ISBN-13
    digits = [int(d) for d in str(isbn) if d.isdigit()]
    if len(digits) != 13:
        return False, "ISBN-13 必须有13位数字"

    total = sum(d * (1 if i % 2 == 0 else 3) for i, d in enumerate(digits))
    is_valid = total % 10 == 0

    if is_valid:
        return True, f"✅ ISBN {isbn} 有效！(总和 {total} 能被10整除)"
    else:
        return False, f"❌ ISBN {isbn} 无效！(总和 {total} 不能被10整除)"

# 计算示例
print("【示例：计算 ISBN 校验位】")
check, msg = isbn13_check_digit("978030640615")
print(f"\n结果: 校验位 = {check}")

# 验证示例
print("\n" + "=" * 60)
print("【验证 ISBN-13】")
test_isbns = ["9780306406157", "9780306406158", "9787302517597"]
for isbn in test_isbns:
    valid, msg = verify_isbn13(isbn)
    print(f"  {msg}")

#### Luhn 算法（Modulo-10）—— 信用卡校验

Luhn 算法（也叫 Modulo-10 算法）被广泛用于信用卡号码、IMEI 号码等的校验。

**计算步骤**：
1. 从右向左，将 **偶数位**（第2、4、6...位）的数字乘以 2
2. 如果乘以 2 后的结果 > 9，则减去 9
3. 将所有数字相加
4. 如果总和能被 10 整除（mod 10 == 0），则号码有效

**举例**：验证 4539 1488 0343 6467

```
原始数字:    4  5  3  9  1  4  8  8  0  3  4  3  6  4  6  7
位置(从右):  16 15 14 13 12 11 10  9  8  7  6  5  4  3  2  1
偶数位×2:    8  5  6  9  2  4 16  8  0  3  8  3 12  4 12  7
>9则-9:      8  5  6  9  2  4  7  8  0  3  8  3  3  4  3  7

总和 = 8+5+6+9+2+4+7+8+0+3+8+3+3+4+3+7 = 80
80 mod 10 = 0  ✅ 有效!
```

In [ ]:
def luhn_validate(number_str):
    # Luhn 算法验证 (Credit Card Validation)
    digits = [int(d) for d in number_str if d.isdigit()]
    n = len(digits)

    print(f"Luhn 算法验证: {number_str}")
    print("=" * 65)

    # Process from right to left
    total = 0
    processed = []

    print(f"{'位置(从右)':>10s} {'原始':>4s} {'操作':>8s} {'结果':>4s}")
    print("-" * 35)

    for i in range(n):
        idx = n - 1 - i  # index from left
        d = digits[idx]
        pos = i + 1  # position from right

        if pos % 2 == 0:  # even position from right
            doubled = d * 2
            if doubled > 9:
                result = doubled - 9
                op = f"{d}*2={doubled}-9"
            else:
                result = doubled
                op = f"{d}*2"
        else:
            result = d
            op = f"{d}"

        total += result
        processed.append(result)
        print(f"{pos:>10d} {d:>4d} {op:>8s} {result:>4d}")

    print("-" * 35)
    print(f"总和 = {total}")
    print(f"{total} mod 10 = {total % 10}")

    if total % 10 == 0:
        print(f"\n✅ 号码有效！(Valid)")
        return True
    else:
        print(f"\n❌ 号码无效！(Invalid)")
        return False

# 测试
print("【测试 1: 有效的信用卡号】")
luhn_validate("4539 1488 0343 6467")

print("\n" + "━" * 65 + "\n")

print("【测试 2: 修改了一位的无效号码】")
luhn_validate("4539 1488 0343 6468")

In [ ]:
def isbn13_calculator():
    # ISBN-13 校验位计算器
    print("=" * 50)
    print("  📚 ISBN-13 校验位计算器")
    print("=" * 50)

    test_cases = [
        ("978030640615", "Harry Potter (示例)"),
        ("978014028032", "1984 by Orwell (示例)"),
        ("978000735934", "The Hobbit (示例)"),
    ]

    for isbn_12, name in test_cases:
        digits = [int(d) for d in isbn_12]
        total = sum(d * (1 if i % 2 == 0 else 3) for i, d in enumerate(digits))
        check = (10 - (total % 10)) % 10
        full_isbn = isbn_12 + str(check)

        print(f"\n  {name}")
        print(f"  前12位: {isbn_12}")
        print(f"  校验位: {check}")
        print(f"  完整ISBN-13: {full_isbn}")

        # Verify
        all_digits = [int(d) for d in full_isbn]
        verify_total = sum(d * (1 if i % 2 == 0 else 3) for i, d in enumerate(all_digits))
        print(f"  验证: 总和={verify_total}, {verify_total}%10={verify_total%10} {'✅' if verify_total%10==0 else '❌'}")

def luhn_calculator():
    # Luhn 校验位计算器
    print("\n" + "=" * 50)
    print("  💳 Luhn 校验位计算器")
    print("=" * 50)

    # Calculate check digit for a number
    partial = "453914880343646"
    digits = [int(d) for d in partial]

    # To calculate: append 0, run Luhn, check digit = (10 - sum%10) % 10
    temp_digits = digits + [0]
    n = len(temp_digits)
    total = 0
    for i in range(n):
        idx = n - 1 - i
        d = temp_digits[idx]
        if (i + 1) % 2 == 0:
            d *= 2
            if d > 9:
                d -= 9
        total += d

    check_digit = (10 - (total % 10)) % 10
    full_number = partial + str(check_digit)

    print(f"\n  输入数字: {partial}")
    print(f"  计算得校验位: {check_digit}")
    print(f"  完整号码: {full_number}")

    # Verify
    all_d = [int(c) for c in full_number]
    n2 = len(all_d)
    verify = 0
    for i in range(n2):
        idx = n2 - 1 - i
        d = all_d[idx]
        if (i + 1) % 2 == 0:
            d *= 2
            if d > 9:
                d -= 9
        verify += d
    print(f"  验证: 总和={verify}, {verify}%10={verify%10} {'✅' if verify%10==0 else '❌'}")

isbn13_calculator()
luhn_calculator()

## 4. Verification 方法详解 (Verification Methods)

### 4.1 数据输入时的核验

#### Visual Check（目视检查）

用户在提交数据前，**人工检查屏幕上显示的数据** 是否与原始数据一致。

> 📋 **例子**：在网购结算前，检查收货地址是否正确；在发送邮件前，检查收件人地址是否正确。

**优点**：简单直接
**缺点**：人容易犯错，尤其是大量数据时容易疲劳遗漏

#### Double Entry（双重输入）

让用户 **输入同一数据两次**，系统比较两次输入是否一致。

> 📋 **例子**：注册账号时要求输入两次密码、两次邮箱地址。

**优点**：能有效检测打字错误
**缺点**：如果用户两次都犯了同样的错误，就无法检测到

### 4.2 数据传输时的核验

当数据通过网络传输时，电磁干扰、硬件故障等都可能导致比特翻转（bit flip）——原本的 0 变成 1，或 1 变成 0。我们需要自动化的方法来检测这种错误。

## 5. 奇偶校验 (Parity Check)

### 基本概念

奇偶校验是最简单的错误检测方法。在传输数据时，额外添加一个 **校验位（Parity Bit）**，使得数据中 1 的总数满足特定规则。

#### 偶校验 (Even Parity)
- 规则：数据中 1 的总数（包括校验位）必须是 **偶数**
- 如果数据中有奇数个 1，校验位设为 1；有偶数个 1，校验位设为 0

#### 奇校验 (Odd Parity)
- 规则：数据中 1 的总数（包括校验位）必须是 **奇数**
- 如果数据中有偶数个 1，校验位设为 1；有奇数个 1，校验位设为 0

**举例（偶校验）**：

```
原始数据:   1 0 1 1 0 0 1    (1的个数 = 4，已经是偶数)
校验位:     0
传输数据:   1 0 1 1 0 0 1 [0]   (1的总数 = 4 ✅ 偶数)

原始数据:   1 0 1 1 0 1 1    (1的个数 = 5，是奇数)
校验位:     1
传输数据:   1 0 1 1 0 1 1 [1]   (1的总数 = 6 ✅ 偶数)
```

### 错误检测

```
发送:       1 0 1 1 0 0 1 [0]   (1的总数 = 4 ✅)
       ↓ 传输过程中第3位发生了翻转
接收:       1 0 0 1 0 0 1 [0]   (1的总数 = 3 ❌ 不是偶数!)
→ 检测到错误！
```

### 局限性

> ⚠️ **奇偶校验只能检测奇数个比特错误**。如果同时有 2 个比特发生翻转，1 的总数的奇偶性不变，错误就无法被检测到！

```
发送:       1 0 1 1 0 0 1 [0]   (1的总数 = 4 ✅)
       ↓ 2个比特同时翻转
接收:       1 0 0 1 0 1 1 [0]   (1的总数 = 4 ✅) ← 看起来没问题!
→ 错误未被检测到！😱
```

### 奇偶字节 / 块校验 (Parity Byte / Block Parity)

为了增强检测能力，可以使用 **二维奇偶校验**。对多行数据同时计算 **行校验位** 和 **列校验位**。

```
         b7  b6  b5  b4  b3  b2  b1  行校验
行1:      1   0   1   1   0   0   1   [0]
行2:      0   1   1   0   1   0   0   [1]
行3:      1   1   0   1   0   1   1   [1]
行4:      0   0   1   0   1   1   0   [1]
列校验:  [0] [0] [1] [0] [0] [0] [0]

→ 如果某个比特翻转，行校验和列校验都会出错，可以定位到具体的比特位置！
```

In [ ]:
def calculate_parity_bit(data_bits, parity_type="even"):
    # 计算奇偶校验位 (Parity Bit Calculator)
    ones_count = sum(data_bits)

    if parity_type == "even":
        parity_bit = ones_count % 2  # 如果1的个数是奇数，加1变偶数
    else:  # odd
        parity_bit = 1 - (ones_count % 2)  # 如果1的个数是偶数，加1变奇数

    return parity_bit

def check_parity(data_with_parity, parity_type="even"):
    # 检查奇偶校验
    ones_count = sum(data_with_parity)

    if parity_type == "even":
        return ones_count % 2 == 0
    else:
        return ones_count % 2 == 1

def demo_parity():
    # 奇偶校验演示
    print("=" * 60)
    print("  Parity Check Demo (奇偶校验演示)")
    print("=" * 60)

    # Example data
    test_data = [
        [1, 0, 1, 1, 0, 0, 1],
        [1, 0, 1, 1, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1],
    ]

    print("\n【偶校验 (Even Parity)】")
    print(f"{'数据':>20s} | {'1的个数':>6s} | {'校验位':>4s} | {'传输数据':>20s}")
    print("-" * 60)

    for data in test_data:
        ones = sum(data)
        pb = calculate_parity_bit(data, "even")
        full = data + [pb]
        data_str = " ".join(map(str, data))
        full_str = " ".join(map(str, data[:])) + f" [{pb}]"
        print(f"{data_str:>20s} | {ones:>6d} | {pb:>4d} | {full_str:>20s}")

    # Simulate error detection
    print("\n【错误检测演示】")
    original = [1, 0, 1, 1, 0, 0, 1]
    pb = calculate_parity_bit(original, "even")
    sent = original + [pb]
    print(f"  发送数据: {' '.join(map(str, sent))}  (1的总数={sum(sent)}, 偶数 ✅)")

    # Simulate 1-bit error
    received = sent.copy()
    received[2] = 1 - received[2]  # flip bit 2
    print(f"  接收数据: {' '.join(map(str, received))}  (1的总数={sum(received)}, {'偶数 ✅' if sum(received)%2==0 else '奇数 ❌'})")
    valid = check_parity(received, "even")
    print(f"  校验结果: {'通过 ✅' if valid else '失败 ❌ - 检测到传输错误!'}")

    # Simulate 2-bit error
    received2 = sent.copy()
    received2[2] = 1 - received2[2]
    received2[5] = 1 - received2[5]
    print(f"\n  发送数据: {' '.join(map(str, sent))}  (1的总数={sum(sent)})")
    print(f"  接收数据: {' '.join(map(str, received2))}  (2位翻转)")
    print(f"  1的总数={sum(received2)}, {'偶数' if sum(received2)%2==0 else '奇数'}")
    valid2 = check_parity(received2, "even")
    print(f"  校验结果: {'通过 ✅ ← 错误未被检测到! ⚠️' if valid2 else '失败 ❌'}")

demo_parity()

In [ ]:
import numpy as np

def block_parity_demo():
    # 块校验/二维奇偶校验演示
    print("=" * 60)
    print("  Block Parity Demo (二维奇偶校验演示)")
    print("=" * 60)

    # Original data (4 rows x 7 columns)
    data = np.array([
        [1, 0, 1, 1, 0, 0, 1],
        [0, 1, 1, 0, 1, 0, 0],
        [1, 1, 0, 1, 0, 1, 1],
        [0, 0, 1, 0, 1, 1, 0],
    ])

    rows, cols = data.shape

    # Calculate row parity
    row_parity = np.array([sum(data[i]) % 2 for i in range(rows)])

    # Calculate column parity
    col_parity = np.array([sum(data[:, j]) % 2 for j in range(cols)])

    print("\n【原始数据 + 校验位】")
    print("         b7  b6  b5  b4  b3  b2  b1  | 行校验")
    print("        " + "-" * 44)
    for i in range(rows):
        row_str = "  ".join(f"{d:2d}" for d in data[i])
        print(f"  行{i+1}:   {row_str}  |  [{row_parity[i]}]")
    print("        " + "-" * 44)
    col_str = "  ".join(f"[{p}]" for p in col_parity)
    print(f"  列校验: {col_str}")

    # Simulate error
    print("\n【模拟单比特错误】")
    error_row, error_col = 1, 2  # Row 2, Column 3 (0-indexed)
    corrupted = data.copy()
    corrupted[error_row][error_col] = 1 - corrupted[error_row][error_col]

    new_row_parity = np.array([sum(corrupted[i]) % 2 for i in range(rows)])
    new_col_parity = np.array([sum(corrupted[:, j]) % 2 for j in range(cols)])

    print(f"  翻转位置: 行{error_row+1}, 列{error_col+1} ({data[error_row][error_col]} → {corrupted[error_row][error_col]})")
    print("\n         b7  b6  b5  b4  b3  b2  b1  | 行校验  原校验  匹配?")
    print("        " + "-" * 60)
    for i in range(rows):
        row_str = "  ".join(f"{d:2d}" for d in corrupted[i])
        match = "✅" if new_row_parity[i] == row_parity[i] else "❌"
        print(f"  行{i+1}:   {row_str}  |  [{new_row_parity[i]}]     [{row_parity[i]}]     {match}")
    print("        " + "-" * 60)

    col_match_str = ""
    for j in range(cols):
        m = "✅" if new_col_parity[j] == col_parity[j] else "❌"
        col_match_str += f"  {m}"
    print(f"  列校验匹配: {col_match_str}")

    # Find error position
    err_rows = [i for i in range(rows) if new_row_parity[i] != row_parity[i]]
    err_cols = [j for j in range(cols) if new_col_parity[j] != col_parity[j]]

    if err_rows and err_cols:
        print(f"\n  🎯 错误定位: 行{err_rows[0]+1}, 列{err_cols[0]+1}")
        print(f"     可以自动修正此比特! (0→1 或 1→0)")

block_parity_demo()

## 6. 校验和 (Checksum)

### 基本概念

校验和是一种更强大的错误检测方法。发送方在发送数据之前，通过特定算法计算出一个 **校验值**，附加在数据后面一起发送。接收方收到数据后，用同样的算法重新计算校验值，并与收到的校验值进行比较。

```
发送方:
┌─────────────┐      ┌─────────────┐
│  原始数据    │  →   │ Checksum算法 │  →  校验值: 42
└─────────────┘      └─────────────┘
                ↓
        发送: [原始数据] + [42]

接收方:
┌─────────────┐      ┌─────────────┐
│  收到的数据  │  →   │ 同样的算法   │  →  重新计算: 42
└─────────────┘      └─────────────┘
                ↓
        比较: 42 == 42?  ✅ 数据完整!
```

### 简单校验和算法

最简单的校验和就是将所有数据字节相加，取结果的低 8 位（mod 256）。

**举例**：数据 = [72, 101, 108, 108, 111]（即 "Hello" 的 ASCII 码）

```
校验和 = (72 + 101 + 108 + 108 + 111) mod 256
       = 500 mod 256
       = 244
```

> 🧠 **校验和 vs 奇偶校验**：
> - 奇偶校验只检测 **单个比特** 的错误
> - 校验和可以检测 **多个字节** 的错误
> - 但校验和也不是万能的——如果两个字节的错误恰好相互抵消（一个+1，一个-1），校验和就检测不到

In [ ]:
def calculate_checksum(data, mod_value=256):
    # 计算校验和 (Checksum Calculator)
    total = sum(data)
    checksum = total % mod_value
    return checksum

def verify_checksum(data, expected_checksum, mod_value=256):
    # 验证校验和
    actual = calculate_checksum(data, mod_value)
    return actual == expected_checksum, actual

def checksum_demo():
    # 校验和演示
    print("=" * 60)
    print("  Checksum Demo (校验和演示)")
    print("=" * 60)

    # Example: "Hello" in ASCII
    message = "Hello"
    data = [ord(c) for c in message]

    print(f"\n原始消息: '{message}'")
    print(f"ASCII 码: {data}")

    # Calculate step by step
    print("\n计算过程:")
    running_total = 0
    for i, (char, val) in enumerate(zip(message, data)):
        running_total += val
        print(f"  + '{char}' (ASCII {val:3d})  →  累计 = {running_total}")

    checksum = running_total % 256
    print(f"\n校验和 = {running_total} mod 256 = {checksum}")

    # Simulate transmission
    print("\n" + "-" * 60)
    print("【模拟传输】")

    # No error
    print("\n  场景1: 数据完整传输")
    received_data = data.copy()
    valid, actual = verify_checksum(received_data, checksum)
    print(f"  收到数据: {received_data}")
    print(f"  重新计算校验和: {actual}")
    print(f"  与原校验和比较: {actual} == {checksum} → {'✅ 匹配!' if valid else '❌ 不匹配!'}")

    # Single byte error
    print("\n  场景2: 一个字节被改变")
    received_data2 = data.copy()
    received_data2[2] = 109  # 'l'(108) -> 'm'(109)
    valid2, actual2 = verify_checksum(received_data2, checksum)
    print(f"  收到数据: {received_data2} (第3字节: {data[2]}→{received_data2[2]})")
    print(f"  重新计算校验和: {actual2}")
    print(f"  与原校验和比较: {actual2} == {checksum} → {'✅ 匹配!' if valid2 else '❌ 不匹配! 检测到错误!'}")

    # Two errors that cancel out
    print("\n  场景3: 两个字节错误相互抵消 ⚠️")
    received_data3 = data.copy()
    received_data3[1] = received_data3[1] + 5  # +5
    received_data3[3] = received_data3[3] - 5  # -5
    valid3, actual3 = verify_checksum(received_data3, checksum)
    print(f"  收到数据: {received_data3} (第2字节+5, 第4字节-5)")
    print(f"  重新计算校验和: {actual3}")
    print(f"  与原校验和比较: {actual3} == {checksum} → {'✅ 匹配! ← 错误未被检测到! ⚠️' if valid3 else '❌ 不匹配!'}")
    print(f"  📌 这说明简单校验和也有局限性!")

checksum_demo()

## 7. Validation vs Verification 总结对比

| 对比维度 | Validation (验证) | Verification (核验) |
|---------|-------------------|---------------------|
| **目的** | 检查数据是否合理 | 检查数据是否准确 |
| **执行方式** | 计算机自动执行 | 通常需要人工参与 |
| **时机** | 数据输入时 | 数据输入时或传输后 |
| **方法** | Range, Format, Length, Presence, Existence, Limit, Check digit | Visual check, Double entry, Parity, Checksum |
| **问的问题** | "这个值有意义吗?" | "这是正确的值吗?" |
| **能发现的错误** | 不合理的数据 (如年龄=300) | 输入错误、传输错误 |
| **不能发现的错误** | 合理但不正确的数据 (如年龄=25写成52) | 两次都输错的情况 |
| **例子** | 邮编必须是6位数字 | 输入两次邮箱地址并比较 |

### 错误检测方法对比

| 方法 | 检测能力 | 复杂度 | 应用场景 |
|------|---------|--------|---------|
| 奇偶校验 (Parity) | 检测奇数个比特错误 | 低 | 内存、简单通信 |
| 校验和 (Checksum) | 检测大部分字节错误 | 中 | 网络传输、文件完整性 |
| 校验位 (Check Digit) | 检测转录和交换错误 | 中 | ISBN、信用卡、身份证 |
| CRC (循环冗余校验) | 检测突发错误 | 高 | 网络协议、存储设备 |
| Hash (哈希) | 检测任何修改 | 高 | 文件完整性、数字签名 |

> 🧠 **考试技巧**：当题目问 "How can you ensure data integrity?" 时，需要根据上下文判断应该用 Validation 还是 Verification：
> - 如果是关于 **数据输入** 的场景 → 考虑 Validation 方法 + Double Entry
> - 如果是关于 **数据传输** 的场景 → 考虑 Parity、Checksum
> - 很多情况下，**两者都需要**！

## 8. 练习题 (Practice Exercises)

### 选择题

**Q1.** 以下哪项是 Validation 而不是 Verification？
- A) 让用户输入两次密码并比较
- B) 检查输入的年龄是否在 0-150 之间
- C) 用户检查屏幕上的信息是否正确
- D) 通过奇偶校验检测传输错误

**Q2.** ISBN-13 中的校验位用于：
- A) 加密书号
- B) 标识出版社
- C) 检测转录和输入错误
- D) 确认书籍真伪

**Q3.** 偶校验中，数据 `1011001` 的校验位应该是？
- A) 0
- B) 1
- C) 2
- D) 无法确定

**Q4.** 以下哪种情况奇偶校验 **无法** 检测到错误？
- A) 1 个比特翻转
- B) 2 个比特翻转
- C) 3 个比特翻转
- D) 5 个比特翻转

**Q5.** Validation 可以确保数据是正确的。这个说法正确吗？为什么？

---

### 编程题

**Q6.** 编写一个函数，对日期进行格式验证（DD/MM/YYYY），包括：
- Format check（格式必须是 DD/MM/YYYY）
- Range check（月份 1-12，日期根据月份决定）
- Length check（必须是 10 个字符）

**Q7.** 编写一个奇偶校验检测程序，能够：
- 对一组 8 位二进制数据添加偶校验位
- 模拟传输中的随机错误
- 检测并报告错误

---

### 简答题

**Q8.** 解释为什么一个在线购物网站在用户注册时需要同时使用 Validation 和 Verification。给出每种方法的具体例子。[4分]

**Q9.** 一家银行需要确保客户通过网上银行转账时数据的完整性。描述可以使用的两种不同的方法，并解释每种方法如何工作。[4分]

**Q10.** 比较奇偶校验（Parity Check）和校验和（Checksum）作为错误检测方法的优缺点。[4分]

---

### 参考答案

<details>
<summary>点击查看选择题答案</summary>

- Q1: **B** - 范围检查是 Validation（计算机自动检查数据是否合理）
- Q2: **C** - 校验位的目的是检测人工输入或转录中的错误
- Q3: **A** - 数据 1011001 中有 4 个 1（偶数），偶校验需要总数为偶数，所以校验位 = 0
- Q4: **B** - 偶数个比特翻转不改变 1 的奇偶性，因此无法被检测
- Q5: **不正确**。Validation 只能检查数据是否合理（reasonable），但不能保证数据是正确的（correct）。例如，年龄 52 通过了范围检查，但用户实际想输入的是 25。

</details>

<details>
<summary>点击查看简答题参考答案</summary>

**Q8:**
Validation 例子：
- 检查邮箱格式是否包含 @ 符号（Format check）[1分]
- 检查密码长度是否至少 8 位（Length check）[1分]

Verification 例子：
- 让用户输入两次密码确保一致（Double entry）[1分]
- 发送验证邮件让用户确认邮箱地址（也是 Verification）[1分]

**Q9:**
- 校验和（Checksum）：在发送转账数据前计算校验和并附加，接收端重新计算并比较 [2分]
- 加密 + 数字签名：使用加密保护数据不被篡改，数字签名验证发送者身份 [2分]

**Q10:**
奇偶校验：
- 优点：实现简单，计算开销小 [1分]
- 缺点：只能检测奇数个比特错误 [1分]

校验和：
- 优点：能检测更多类型的错误，适用于更大的数据块 [1分]
- 缺点：如果错误恰好抵消则无法检测，计算开销比奇偶校验大 [1分]

</details>

## 📋 本节总结 (Chapter Summary)

### 核心知识点

1. **数据完整性**：确保数据在整个生命周期中保持准确、一致、可靠

2. **Validation vs Verification**（最重要的区别！）：
   - Validation = 计算机自动检查数据是否 **合理**
   - Verification = 确认数据是否是用户 **真正想要的**

3. **Validation 方法**：
   - Range Check（范围检查）
   - Format Check（格式检查）
   - Length Check（长度检查）
   - Presence Check（非空检查）
   - Existence Check（查找检查）
   - Limit Check（限值检查）
   - Check Digit（校验位）：ISBN-13、Luhn 算法

4. **Verification 方法**：
   - 数据输入：Visual Check（目视检查）、Double Entry（双重输入）
   - 数据传输：Parity Check（奇偶校验）、Checksum（校验和）

5. **奇偶校验**：通过校验位使 1 的总数满足奇偶规则，只能检测奇数个比特错误

6. **校验和**：计算数据的总和取模，能检测大部分传输错误

### 考试关键词
`Data Integrity` `Validation` `Verification` `Range Check` `Format Check`
`Length Check` `Presence Check` `Existence Check` `Limit Check` `Check Digit`
`ISBN-13` `Luhn Algorithm` `Parity Check` `Even Parity` `Odd Parity`
`Parity Byte` `Checksum` `Visual Check` `Double Entry`

---
> 📚 **恭喜完成第6章！** 你已经学会了数据安全与数据完整性的核心知识。记住：**Security 保护数据不被偷，Integrity 保证数据不出错。两者缺一不可！**